# AI / LLM Scientific Review Framework — Evaluation Notebook

This notebook evaluates uploaded documents (PDF, DOCX, Excel) against the **AI/LLM Scientific Review Framework v1.0**.

**Five evaluation domains:** Accuracy, Safety, Transparency, Repeatability, Trustworthiness  
**Plus:** Modern Design Pattern checklist (6 pillars) and Risk Register coverage  

---

### How to use
1. **Option A** — Place documents in the Dataiku managed folder `eval_documents`
2. **Option B** — Upload a file using the widget in Cell 3
3. Run all cells to generate the evaluation report

In [ ]:
# Cell 1: Imports and setup
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add project lib to path (works in Dataiku and locally)
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from IPython.display import display, HTML

from lib.document_parser import parse_file, SUPPORTED_EXTENSIONS
from lib.evaluator import run_full_evaluation
from lib.report_generator import display_full_report, results_to_dataframe

print(f"Supported file types: {', '.join(sorted(SUPPORTED_EXTENSIONS))}")
print("Ready.")

In [ ]:
# Cell 2: Load documents from Dataiku managed folder OR local directory
#
# In Dataiku: uses dataiku.Folder("eval_documents")
# Locally:    falls back to ./eval_documents/ directory

documents = {}  # filename -> raw bytes

try:
    import dataiku
    folder = dataiku.Folder("eval_documents")
    file_list = folder.list_paths_in_partition()
    for path in file_list:
        ext = os.path.splitext(path)[1].lower()
        if ext in SUPPORTED_EXTENSIONS:
            with folder.get_download_stream(path) as stream:
                documents[os.path.basename(path)] = stream.read()
    print(f"Loaded {len(documents)} document(s) from Dataiku folder 'eval_documents'")
except (ImportError, Exception) as e:
    # Fallback: local directory
    local_folder = os.path.join(PROJECT_ROOT, "eval_documents")
    if os.path.isdir(local_folder):
        for fname in os.listdir(local_folder):
            ext = os.path.splitext(fname)[1].lower()
            if ext in SUPPORTED_EXTENSIONS:
                with open(os.path.join(local_folder, fname), "rb") as f:
                    documents[fname] = f.read()
        print(f"Loaded {len(documents)} document(s) from local folder '{local_folder}'")
    else:
        print(f"No 'eval_documents' folder found. Use the upload widget in the next cell.")

if documents:
    for name in documents:
        print(f"  - {name} ({len(documents[name]):,} bytes)")

In [ ]:
# Cell 3: Upload widget (alternative to managed folder)
# Works in Jupyter / Dataiku notebook environments

try:
    import ipywidgets as widgets

    upload_widget = widgets.FileUpload(
        accept=".pdf,.docx,.xlsx,.xls",
        multiple=True,
        description="Upload Documents",
    )

    upload_output = widgets.Output()

    def on_upload_change(change):
        with upload_output:
            upload_output.clear_output()
            for uploaded_file in upload_widget.value:
                name = uploaded_file["name"] if isinstance(uploaded_file, dict) else uploaded_file.name
                content = uploaded_file["content"] if isinstance(uploaded_file, dict) else uploaded_file.content
                # Handle memoryview
                if isinstance(content, memoryview):
                    content = bytes(content)
                documents[name] = content
                print(f"Uploaded: {name} ({len(content):,} bytes)")

    upload_widget.observe(on_upload_change, names="value")
    display(widgets.VBox([upload_widget, upload_output]))

except ImportError:
    print("ipywidgets not available. Place files in 'eval_documents' folder instead.")

In [ ]:
# Cell 4: Parse and evaluate all documents

if not documents:
    display(HTML(
        '<div style="background:#fff3e0;border:1px solid #e65100;border-radius:6px;padding:16px;color:#e65100">'
        '<strong>No documents loaded.</strong> Please add files to the eval_documents folder '
        'or use the upload widget above, then re-run this cell.</div>'
    ))
else:
    all_results = {}
    for filename, file_bytes in documents.items():
        print(f"Parsing: {filename} ...", end=" ")
        try:
            text = parse_file(filename, file_bytes)
            word_count = len(text.split())
            print(f"{word_count:,} words extracted.")
            results = run_full_evaluation(text)
            all_results[filename] = {"text": text, "results": results}
        except Exception as exc:
            print(f"ERROR: {exc}")
    
    print(f"\nEvaluated {len(all_results)} document(s) successfully.")

In [ ]:
# Cell 5: Display evaluation reports

for filename, data in all_results.items():
    display_full_report(filename, data["results"])
    display(HTML("<br><hr style='border:2px solid #ccc'><br>"))

In [ ]:
# Cell 6: Export results as a DataFrame (can be written to Dataiku dataset)

all_dfs = []
for filename, data in all_results.items():
    df = results_to_dataframe(data["results"])
    df.insert(0, "document", filename)
    all_dfs.append(df)

if all_dfs:
    results_df = pd.concat(all_dfs, ignore_index=True)
    display(HTML("<h3>Results DataFrame (exportable)</h3>"))
    display(results_df)
else:
    print("No results to export.")

In [ ]:
# Cell 7 (Optional): Write results to a Dataiku output dataset
# Uncomment and set dataset name to enable

# import dataiku
# output_dataset = dataiku.Dataset("eval_results")
# output_dataset.write_with_schema(results_df)

In [ ]:
# Cell 8: Multi-document comparison (if >1 document evaluated)

if len(all_results) > 1:
    comparison_rows = []
    for filename, data in all_results.items():
        r = data["results"]
        row = {"Document": filename}
        for domain, ddata in r["domains"].items():
            row[domain] = ddata["score"]
        row["Composite"] = r["composite"]["composite_score"]
        row["Verdict"] = r["composite"]["verdict"]
        row["Design Patterns"] = r["design_patterns_overall_score"]
        row["Risks Addressed"] = f"{r['risks_addressed']}/{r['risks_total']}"
        comparison_rows.append(row)

    comp_df = pd.DataFrame(comparison_rows)
    display(HTML("<h3>Multi-Document Comparison</h3>"))
    display(HTML(comp_df.style.hide(axis='index').to_html()))
else:
    print("Upload multiple documents to see a comparison table.")